# Fever Forecasting with Chronos-2

This notebook loads long-format fever time series data, creates a per-series holdout split (last `prediction_length` points),
runs Chronos-2 forecasting, and saves predictions.

**Input format (long):**
- `encounter_id`
- `recorded_time`
- `value`

Chronos expects:
- `item_id`
- `timestamp`
- `target`


In [ ]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd

from chronos import BaseChronosPipeline, Chronos2Pipeline

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
LOGGER = logging.getLogger("chronos2_notebook")


In [ ]:
# Paths
DATA_DIR = Path("./YOUR_PATH")  # change to your folder
TS_CSV = DATA_DIR / "YOUR_FILE.csv"  # your long-format time series
TS_SEP = ";"  # adjust

# Chronos settings
MODEL_ID = "amazon/chronos-2"
DEVICE_MAP = "auto"        # "cuda" / "cpu" / "auto"
PREDICTION_LENGTH = 3      # horizon length in rows/steps (not hours)
QUANTILES = [0.1, 0.5, 0.9]

# Outputs
OUT_DIR = Path("./YOUR_PATH")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PREDICTIONS = OUT_DIR / "chronos2_predictions.csv"

# merge max forecast into your tabular coviarate table
STATIC_CSV = None  # e.g. DATA_DIR / "YOUR_TABFILE.csv"
STATIC_SEP = ";"
STATIC_ID_COL = "YOUR_ID_COLUMN"
OUT_STATIC_MERGED = OUT_DIR / "YOUR_TABFILE_WITH_FORECASTS.csv"


In [ ]:
def require_columns(df: pd.DataFrame, cols: list[str], name: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns: {missing}")


def load_long_timeseries(
    path: Path,
    *,
    sep: str = ";",
    id_col: str = "encounter_id",
    time_col: str = "recorded_time",
    value_col: str = "value",
) -> pd.DataFrame:
    df = pd.read_csv(path, sep=sep)
    require_columns(df, [id_col, time_col, value_col], name=f"timeseries({path.name})")

    out = df.rename(columns={id_col: "item_id", time_col: "timestamp", value_col: "target"}).copy()
    out["item_id"] = out["item_id"].astype(str)
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True, errors="coerce")
    out["target"] = pd.to_numeric(out["target"], errors="coerce")

    # Drop invalid rows
    out = out.dropna(subset=["item_id", "timestamp", "target"])

    # Deterministic order
    out = out.sort_values(["item_id", "timestamp"]).reset_index(drop=True)
    return out


def split_last_horizon_per_item(
    df: pd.DataFrame,
    *,
    prediction_length: int,
    id_col: str = "item_id",
    time_col: str = "timestamp",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Hold out the last `prediction_length` rows per item as test.
    Drops items shorter than or equal to prediction_length.
    """
    require_columns(df, [id_col, time_col, "target"], name="df_for_split")

    train_parts = []
    test_parts = []

    for _, g in df.groupby(id_col, sort=False):
        g = g.sort_values(time_col)
        if len(g) <= prediction_length:
            continue
        test_parts.append(g.tail(prediction_length))
        train_parts.append(g.iloc[:-prediction_length])

    if not train_parts:
        raise ValueError("No series long enough to split. Reduce prediction_length or check data.")

    return pd.concat(train_parts, ignore_index=True), pd.concat(test_parts, ignore_index=True)

def split_by_anchor_time(
    df: pd.DataFrame,
    *,
    id_col: str = "item_id",
    time_col: str = "timestamp",
    target_col: str = "target",
    anchor_col: str = "antibiotic_start_time",
    context_hours: int = 48,
    forecast_hours: int = 24,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Leakage-safe split for downstream classification.

    Train/context:
        timestamps <= anchor + context_hours

    Test/future:
        anchor + context_hours < timestamps <= anchor + context_hours + forecast_hours
    """
    require_columns(df, [id_col, time_col, target_col, anchor_col], name="df_for_anchor_split")

    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], utc=True)
    out[anchor_col] = pd.to_datetime(out[anchor_col], utc=True)

    out["context_cutoff"] = out[anchor_col] + pd.Timedelta(hours=context_hours)
    out["forecast_start"] = out["context_cutoff"]
    out["forecast_end"] = out["context_cutoff"] + pd.Timedelta(hours=forecast_hours)

    train_df = out.loc[out[time_col] <= out["context_cutoff"]].copy()

    test_df = out.loc[
        (out[time_col] > out["forecast_start"]) &
        (out[time_col] <= out["forecast_end"])
    ].copy()

    if train_df.empty:
        raise ValueError("No context rows found. Check anchor times and context_hours.")

    if test_df.empty:
        raise ValueError("No forecast-window rows found. Check anchor times and forecast_hours.")

    assert (train_df[time_col] <= train_df["context_cutoff"]).all(), (
        "Leakage detected: context contains rows after cutoff."
    )

    return train_df, test_df

def max_forecast_per_item(pred_df: pd.DataFrame) -> pd.DataFrame:
    require_columns(pred_df, ["item_id", "predictions"], name="pred_df")
    return (
        pred_df.groupby("item_id", as_index=False)
        .agg(predictions_max=("predictions", "max"))
    )


In [ ]:
LOGGER.info("Loading time series from %s", TS_CSV)
context_df = load_long_timeseries(TS_CSV, sep=TS_SEP)

LOGGER.info("Loaded %d rows across %d series", len(context_df), context_df["item_id"].nunique())
context_df.head()

#train_df, test_df = split_last_horizon_per_item(context_df, prediction_length=PREDICTION_LENGTH)

train_df, test_df = split_by_anchor_time(
    df_for_split,
    id_col="item_id",
    time_col="timestamp",
    target_col="target",
    anchor_col="antibiotic_start_time",
    context_hours=48,
    forecast_hours=24,
)

LOGGER.info("Train rows=%d | Test rows=%d", len(train_df), len(test_df))
train_df.head()

LOGGER.info("Loading Chronos-2 pipeline: %s (device_map=%s)", MODEL_ID, DEVICE_MAP)
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained(MODEL_ID, device_map=DEVICE_MAP)


In [ ]:
LOGGER.info("Forecasting with prediction_length=%d", PREDICTION_LENGTH)

pred_df = pipeline.predict_df(
    train_df,
    prediction_length=PREDICTION_LENGTH,
    quantile_levels=QUANTILES,
    id_column="item_id",
    timestamp_column="timestamp",
    target="target",
)

pred_df.head()


In [ ]:
def evaluate_median_mae(pred_df: pd.DataFrame, test_df: pd.DataFrame) -> pd.DataFrame:
    """
    Evaluate median (q=0.5) MAE against held-out horizon.
    Assumes pred_df includes columns: item_id, timestamp, quantile, predictions
    (Chronos naming may include 'quantile' or 'quantile_level' depending on version.)
    """
    # Make this robust to quantile column naming
    q_col = "quantile" if "quantile" in pred_df.columns else ("quantile_level" if "quantile_level" in pred_df.columns else None)
    if q_col is None:
        raise ValueError(f"Cannot find quantile column in pred_df. Columns: {pred_df.columns.tolist()}")

    median_pred = pred_df.loc[pred_df[q_col] == 0.5, ["item_id", "timestamp", "predictions"]].copy()

    # Align to actual held-out values
    actual = test_df[["item_id", "timestamp", "target"]].copy()
    merged = actual.merge(median_pred, on=["item_id", "timestamp"], how="inner")

    if merged.empty:
        raise ValueError("No overlap between predictions and test timestamps. Check timestamp alignment.")

    merged["abs_err"] = (merged["predictions"] - merged["target"]).abs()

    per_item = merged.groupby("item_id", as_index=False).agg(
        mae=("abs_err", "mean"),
        n=("abs_err", "size"),
    )
    overall = pd.DataFrame([{
        "mae": merged["abs_err"].mean(),
        "n": len(merged),
        "items": merged["item_id"].nunique(),
    }])

    return per_item.sort_values("mae", ascending=False), overall


per_item_mae, overall_mae = evaluate_median_mae(pred_df, test_df)
overall_mae

LOGGER.info("Saving predictions to %s", OUT_PREDICTIONS)
pred_df.to_csv(OUT_PREDICTIONS, index=False)
OUT_PREDICTIONS


In [ ]:
if STATIC_CSV is not None:
    LOGGER.info("Loading static dataset from %s", STATIC_CSV)
    static_df = pd.read_csv(STATIC_CSV, sep=STATIC_SEP)
    require_columns(static_df, [STATIC_ID_COL], name="static_df")

    max_pred = max_forecast_per_item(pred_df)

    # Normalize merge keys to string
    static_df["_merge_id"] = static_df[STATIC_ID_COL].astype(str)
    max_pred["_merge_id"] = max_pred["item_id"].astype(str)

    merged = static_df.merge(max_pred.drop(columns=["item_id"]), on="_merge_id", how="left").drop(columns=["_merge_id"])

    LOGGER.info("Saving merged static dataset to %s", OUT_STATIC_MERGED)
    merged.to_csv(OUT_STATIC_MERGED, sep=STATIC_SEP, index=False)

    OUT_STATIC_MERGED
else:
    LOGGER.info("STATIC_CSV is None; skipping static merge.")
